# BreatheCast AQI Prediction — Explained ML Pipeline

This notebook trains the BreatheCast AQI prediction model while also showing **why each preprocessing step is used**.

The production modeling pipeline is preserved: the final estimator remains an **XGBoost Regressor** wrapped in a **Yeo–Johnson target transformation**, with the same feature groups, 80/20 split, random seed, and XGBoost hyperparameters used by the current BreatheCast artifact.

The additional charts are for understanding and diagnosis only. They do **not** change the final production pipeline.

## Notebook goals

1. Inspect the dataset and target distribution.
2. Visualize feature distributions and relationships with AQI.
3. Use graphs to motivate the selected preprocessing transformations.
4. Train the same final model pipeline.
5. Evaluate predictions with regression metrics and diagnostic plots.
6. Inspect feature importance.
7. Export the model artifact expected by BreatheCast.


## 1. Imports

Three visualization libraries are used for different purposes:

- **Matplotlib** — precise static ML diagnostic plots such as actual-vs-predicted and residuals.
- **Seaborn** — compact statistical plots such as distributions, boxplots, and correlation heatmaps.
- **Plotly Express** — interactive plots that make relationships easier to explore when viewing the notebook.

The visualization cells are intentionally separate from the training code.


In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, QuantileTransformer

from xgboost import XGBRegressor


## 2. Load the dataset

The current training workflow expects the dataset at `../data/weather_aqi_data.csv`. Change `DATA_PATH` only when the file is stored elsewhere.


In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("khawajazainali/synthetic-aqi-prediction-dataset")

dataset = pd.read_csv(os.path.join(path, "weather_aqi_data.csv"))

print(f"Dataset shape: {dataset.shape}")
display(dataset.head())

## 3. Basic data audit

Before modeling, check data types, missing values, duplicates, and the target range. This gives a quick view of whether preprocessing is needed and whether the target looks suitable for regression.


In [ ]:
print("Data types:")
display(dataset.dtypes.to_frame("dtype"))

missing = dataset.isna().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0].to_frame("missing_count"))

print(f"Duplicate rows: {dataset.duplicated().sum():,}")
print(f"Target range: {dataset['AQI_Value'].min()} to {dataset['AQI_Value'].max()}")


## 4. Define model features and target

The final model uses 12 input features and predicts the numeric `AQI_Value` target.


In [ ]:
FEATURES = [
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "Temperature_C",
    "Humidity_pct",
    "WindSpeed_kmh",
    "WindDirection_deg",
    "Pressure_hPa",
    "SolarRadiation_Wm2",
    "Rainfall_mm",
    "TrafficDensityIndex",
    "ProximityIndustrialZone_km",
]

TARGET = "AQI_Value"

required_columns = FEATURES + [TARGET]
missing_columns = [column for column in required_columns if column not in dataset.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = dataset[required_columns].copy()
print(f"Rows available: {len(df):,}")
print(f"Features: {len(FEATURES)}")


## 5. Target distribution — why this is a regression problem

The target is numeric, so the model predicts a continuous AQI value instead of selecting a class label. The distribution plot also helps us see the scale and shape of the target before training.


In [ ]:
fig = px.histogram(
    df,
    x=TARGET,
    nbins=60,
    marginal="box",
    title="AQI Value Distribution",
    labels={TARGET: "AQI Value"},
)
fig.show()


## 6. Numeric feature distributions

The preprocessing is intentionally different for different numeric features. Before applying transformations, inspect their shapes. Features with strong skew or long tails are candidates for the transformations used later.

The four plots below focus on the features for which the pipeline uses **PowerTransformer** or **QuantileTransformer**.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=df, x="Temperature_C", kde=True, ax=ax)
ax.set_title("Temperature Distribution")
ax.set_xlabel("Temperature_C")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=df, x="WindSpeed_kmh", kde=True, ax=ax)
ax.set_title("Wind Speed Distribution")
ax.set_xlabel("WindSpeed_kmh")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=df, x="TrafficDensityIndex", kde=True, ax=ax)
ax.set_title("Traffic Density Distribution")
ax.set_xlabel("TrafficDensityIndex")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=df, x="ProximityIndustrialZone_km", kde=True, ax=ax)
ax.set_title("Industrial Proximity Distribution")
ax.set_xlabel("ProximityIndustrialZone_km")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


## 7. Numeric feature skewness

Skewness gives a compact numerical summary of distribution asymmetry. It is not the only reason to transform a feature, but it helps explain why selected variables receive stronger distributional preprocessing.


In [ ]:
numeric_columns = [
    "Temperature_C",
    "Humidity_pct",
    "WindSpeed_kmh",
    "WindDirection_deg",
    "Pressure_hPa",
    "SolarRadiation_Wm2",
    "Rainfall_mm",
    "TrafficDensityIndex",
    "ProximityIndustrialZone_km",
]

skewness = df[numeric_columns].skew(numeric_only=True).sort_values(key=np.abs, ascending=False)
display(skewness.to_frame("skewness"))


## 8. Correlation with AQI

A correlation heatmap helps us understand which numeric variables have linear association with the target. This is **diagnostic only**: XGBoost can learn nonlinear relationships and interactions, so a low Pearson correlation does not mean a feature is useless.


In [ ]:
corr_columns = numeric_columns + [TARGET]
correlation = df[corr_columns].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
)
ax.set_title("Correlation Heatmap: Numeric Features and AQI")
plt.tight_layout()
plt.show()


## 9. Feature vs AQI relationships

These interactive scatter plots show why a tree-based model is a reasonable choice: the relationship between an environmental variable and AQI does not have to be purely linear. The model can learn nonlinear thresholds and feature interactions.


In [ ]:
fig = px.scatter(
    df,
    x="Temperature_C",
    y=TARGET,
    opacity=0.35,
    render_mode="webgl",
    title="Temperature_C vs AQI",
    labels={TARGET: "AQI Value"},
)
fig.show()


In [ ]:
fig = px.scatter(
    df,
    x="WindSpeed_kmh",
    y=TARGET,
    opacity=0.35,
    render_mode="webgl",
    title="WindSpeed_kmh vs AQI",
    labels={TARGET: "AQI Value"},
)
fig.show()


In [ ]:
fig = px.scatter(
    df,
    x="TrafficDensityIndex",
    y=TARGET,
    opacity=0.35,
    render_mode="webgl",
    title="TrafficDensityIndex vs AQI",
    labels={TARGET: "AQI Value"},
)
fig.show()


In [ ]:
fig = px.scatter(
    df,
    x="ProximityIndustrialZone_km",
    y=TARGET,
    opacity=0.35,
    render_mode="webgl",
    title="ProximityIndustrialZone_km vs AQI",
    labels={TARGET: "AQI Value"},
)
fig.show()


## 10. Categorical features

`DayOfWeek`, `Month`, and `IsWeekend` are categorical/time-context variables. One-hot encoding lets the model treat categories independently instead of interpreting labels as continuous numeric distances.


In [ ]:
print("Day of week counts")
display(df["DayOfWeek"].value_counts(dropna=False).rename_axis("DayOfWeek").to_frame("count"))

fig = px.bar(
    df["DayOfWeek"].value_counts(dropna=False).rename_axis("DayOfWeek").reset_index(name="count"),
    x="DayOfWeek",
    y="count",
    title="Observations by Day of Week",
)
fig.show()


In [ ]:
fig = px.bar(
    df["Month"].value_counts(dropna=False).rename_axis("Month").reset_index(name="count"),
    x="Month",
    y="count",
    title="Observations by Month",
)
fig.show()


In [ ]:
weekend_counts = (
    df["IsWeekend"]
    .map({0: "Weekday", 1: "Weekend", False: "Weekday", True: "Weekend"})
    .value_counts()
    .rename_axis("day_type")
    .reset_index(name="count")
)

fig = px.bar(
    weekend_counts,
    x="day_type",
    y="count",
    title="Weekday vs Weekend Observations",
)
fig.show()


## 11. Convert categorical values

The stored dataset uses numeric representations for month and day-of-week. We convert those codes to category names before one-hot encoding. `IsWeekend` is retained as a categorical boolean feature.


In [ ]:
MONTH_MAP = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December",
}

DAY_OF_WEEK_MAP = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday",
    3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday",
}

df["Month"] = df["Month"].map(MONTH_MAP)
df["DayOfWeek"] = df["DayOfWeek"].map(DAY_OF_WEEK_MAP)
df["IsWeekend"] = df["IsWeekend"].astype(bool)

CATEGORICAL_COLUMNS = ["DayOfWeek", "Month"]

print("Categorical columns:", CATEGORICAL_COLUMNS + ["IsWeekend"])


## 12. Why each preprocessing branch exists

### Power transformation — `Temperature_C`, `WindSpeed_kmh`

A Yeo–Johnson transform is useful when a numeric feature is noticeably skewed but we still want a smooth continuous transformation. Unlike a simple log transform, Yeo–Johnson can handle zero and negative values.

### Quantile transformation — `TrafficDensityIndex`, `ProximityIndustrialZone_km`

These variables can contain concentrated ranges and long tails. `QuantileTransformer(output_distribution="normal")` reshapes their empirical distribution toward a normal-like distribution, reducing the impact of extreme concentration/tails.

### No distribution transform — other numeric features

These features are only median-imputed in the final pipeline. XGBoost is a tree-based model and does not require feature scaling in the way distance-based or gradient-based linear models often do.

### One-hot encoding — categorical features

Days, months, and weekend status are categories rather than measurements with meaningful numeric distance. One-hot encoding prevents an artificial ordering from being imposed.


## 13. Visualize the selected transformations

These plots directly demonstrate the effect of the same transformations used inside the training pipeline.

**Important:** transformations are fit using the training data only. This avoids leaking information from the test set into preprocessing.


In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")


In [ ]:
# Fit the same individual transforms on training data for visualization only.
power_columns = ["Temperature_C", "WindSpeed_kmh"]
quantile_columns = ["TrafficDensityIndex", "ProximityIndustrialZone_km"]

power_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("power", PowerTransformer(method="yeo-johnson", standardize=True)),
])

quantile_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("quantile", QuantileTransformer(output_distribution="normal", random_state=42)),
])

power_train = pd.DataFrame(
    power_transformer.fit_transform(X_train[power_columns]),
    columns=power_columns,
    index=X_train.index,
)

quantile_train = pd.DataFrame(
    quantile_transformer.fit_transform(X_train[quantile_columns]),
    columns=quantile_columns,
    index=X_train.index,
)


In [ ]:
fig, axes = plt.subplots(figsize=(9, 4.5))
sns.histplot(X_train["Temperature_C"], kde=True, stat="density", label="Before", alpha=0.35, ax=axes)
sns.histplot(power_train["Temperature_C"], kde=True, stat="density", label="After Yeo–Johnson", alpha=0.35, ax=axes)
axes.set_title("Temperature_C: Before vs After Yeo–Johnson")
axes.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(figsize=(9, 4.5))
sns.histplot(X_train["WindSpeed_kmh"], kde=True, stat="density", label="Before", alpha=0.35, ax=axes)
sns.histplot(power_train["WindSpeed_kmh"], kde=True, stat="density", label="After Yeo–Johnson", alpha=0.35, ax=axes)
axes.set_title("WindSpeed_kmh: Before vs After Yeo–Johnson")
axes.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(figsize=(9, 4.5))
sns.histplot(X_train["TrafficDensityIndex"], kde=True, stat="density", label="Before", alpha=0.35, ax=axes)
sns.histplot(quantile_train["TrafficDensityIndex"], kde=True, stat="density", label="After Quantile Transform", alpha=0.35, ax=axes)
axes.set_title("TrafficDensityIndex: Before vs After Quantile Transform")
axes.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(figsize=(9, 4.5))
sns.histplot(X_train["ProximityIndustrialZone_km"], kde=True, stat="density", label="Before", alpha=0.35, ax=axes)
sns.histplot(quantile_train["ProximityIndustrialZone_km"], kde=True, stat="density", label="After Quantile Transform", alpha=0.35, ax=axes)
axes.set_title("ProximityIndustrialZone_km: Before vs After Quantile Transform")
axes.legend()
plt.tight_layout()
plt.show()


## 14. Build the production preprocessing pipeline

The implementation below is the same production preprocessing strategy used by BreatheCast. The visual analysis above explains the *reason* for the branch choices; this cell defines the actual pipeline used for training.


In [ ]:
def build_preprocessor(
    categorical_columns: list[str],
    use_scaling: bool = True,
) -> ColumnTransformer:
    power_columns = [
        "Temperature_C",
        "WindSpeed_kmh",
    ]

    normal_numeric_columns = [
        "Rainfall_mm",
        "SolarRadiation_Wm2",
        "Pressure_hPa",
        "Humidity_pct",
        "WindDirection_deg",
    ]

    quantile_columns = [
        "TrafficDensityIndex",
        "ProximityIndustrialZone_km",
    ]

    categorical_columns = categorical_columns.copy() + ["IsWeekend"]

    power_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "power",
            PowerTransformer(
                method="yeo-johnson",
                standardize=use_scaling,
            ),
        ),
    ])

    normal_numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    quantile_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "quantile",
            QuantileTransformer(
                output_distribution="normal",
                random_state=42,
            ),
        ),
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ])

    return ColumnTransformer([
        ("power", power_transformer, power_columns),
        ("normal", normal_numeric_transformer, normal_numeric_columns),
        ("quantile", quantile_transformer, quantile_columns),
        ("cat", categorical_transformer, categorical_columns),
    ], remainder="drop")


def build_pipeline(
    model: BaseEstimator,
    preprocessor: ColumnTransformer | None = None,
) -> Pipeline:
    steps = []
    if preprocessor is not None:
        steps.append(("preprocessor", preprocessor))
    steps.append(("model", model))
    return Pipeline(steps)


## 15. Build the final XGBoost model

The final estimator and hyperparameters are kept unchanged. The target is transformed with Yeo–Johnson using `TransformedTargetRegressor`, so predictions are automatically returned on the original AQI scale.


In [ ]:
preprocessor = build_preprocessor(
    categorical_columns=CATEGORICAL_COLUMNS,
    use_scaling=True,
)

model = TransformedTargetRegressor(
    regressor=XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    ),
    transformer=PowerTransformer(
        method="yeo-johnson",
        standardize=True,
    ),
)

pipeline = build_pipeline(
    model=model,
    preprocessor=preprocessor,
)

pipeline


## 16. Train and evaluate


In [ ]:
def evaluate_regressor(y_true, y_pred) -> dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "r2": r2_score(y_true, y_pred),
    }

pipeline.fit(X_train, y_train)
y_pred_test = pipeline.predict(X_test)
test_metrics = evaluate_regressor(y_test, y_pred_test)

print(test_metrics)


## 17. Actual vs predicted — model fit quality

A strong regression model should place predictions close to the diagonal line `prediction = actual`. This plot gives a more intuitive view of the test-set error than a single metric.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_test, alpha=0.45)
line_min = min(y_test.min(), y_pred_test.min())
line_max = max(y_test.max(), y_pred_test.max())
ax.plot([line_min, line_max], [line_min, line_max], linestyle="--")
ax.set_title("Actual vs Predicted AQI — Test Set")
ax.set_xlabel("Actual AQI")
ax.set_ylabel("Predicted AQI")
plt.tight_layout()
plt.show()


## 18. Residual distribution

Residuals are `actual - predicted`. A centered residual distribution with relatively small spread indicates that most predictions are close to the true target values.


In [ ]:
residuals = y_test.to_numpy() - y_pred_test

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(residuals, kde=True, ax=ax)
ax.axvline(0, linestyle="--")
ax.set_title("Residual Distribution — Test Set")
ax.set_xlabel("Residual (Actual − Predicted)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


## 19. Residuals vs predictions

This interactive diagnostic helps reveal systematic error. Ideally, residuals should not form a strong curve or widening funnel as predictions increase.


In [ ]:
residual_df = pd.DataFrame({
    "Predicted_AQI": y_pred_test,
    "Residual": residuals,
})

fig = px.scatter(
    residual_df,
    x="Predicted_AQI",
    y="Residual",
    opacity=0.45,
    title="Residuals vs Predicted AQI",
)
fig.add_hline(y=0, line_dash="dash")
fig.show()


## 20. Cross-validation

Five-fold cross-validation provides an additional estimate of how consistently the pipeline generalizes across the training data. The final held-out test set remains the primary unbiased evaluation reported for the current model.


In [ ]:
y_pred_cv = cross_val_predict(
    pipeline,
    X_train,
    y_train,
    cv=5,
    n_jobs=-1,
)

cv_metrics = evaluate_regressor(y_train, y_pred_cv)
print(cv_metrics)


## 21. Feature importance

XGBoost can provide feature-importance scores after preprocessing. Because one-hot encoding expands categorical variables into multiple columns, importance is shown at the **transformed feature** level.

This does not prove causality; it only shows which transformed inputs were most useful to the fitted trees.


In [ ]:
preprocessor_fitted = pipeline.named_steps["preprocessor"]
transformed_names = preprocessor_fitted.get_feature_names_out()

transformed_model = pipeline.named_steps["model"].regressor_
importance = transformed_model.feature_importances_

importance_df = (
    pd.DataFrame({"feature": transformed_names, "importance": importance})
    .sort_values("importance", ascending=False)
    .head(20)
    .sort_values("importance")
)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(importance_df["feature"], importance_df["importance"])
ax.set_title("Top 20 Transformed Feature Importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()


## 22. Reference results from the original final training run

The same final pipeline produced these benchmark values in the original training run:

| Metric | Score |
|---|---:|
| MAE | **6.0728** |
| MSE | **66.0523** |
| RMSE | **8.1273** |
| R² | **0.9677** |

Small differences may appear if package versions or the underlying training data differ.


## 23. Create the BreatheCast model artifact

The backend expects a joblib artifact containing the trained pipeline and metadata.


In [ ]:
artifact = {
    "model": pipeline,
    "metadata": {
        "task": "Regression",
        "features": FEATURES,
        "target": TARGET,
        "metrics": test_metrics,
        "model_name": "XGBRegressor",
    },
}

ARTIFACT_PATH = "../backend/app/models/aqi_artifact_1.0.pkl"

joblib.dump(artifact, ARTIFACT_PATH)
print(f"Artifact saved to: {ARTIFACT_PATH}")


## 24. Verify the saved artifact


In [ ]:
loaded_artifact = joblib.load(ARTIFACT_PATH)
loaded_model = loaded_artifact["model"]

print("Metadata:")
print(loaded_artifact["metadata"])

sample_prediction = loaded_model.predict(X_test.iloc[[0]])[0]
print(f"Sample prediction: {sample_prediction:.4f}")


## 25. Model input contract

The BreatheCast backend sends the following model inputs:

```text
DayOfWeek
Month
IsWeekend
Temperature_C
Humidity_pct
WindSpeed_kmh
WindDirection_deg
Pressure_hPa
SolarRadiation_Wm2
Rainfall_mm
TrafficDensityIndex
ProximityIndustrialZone_km
```

`IsWeekend` is normally derived by the backend from `DayOfWeek`.

### Final takeaway

The preprocessing branches are not arbitrary:

- **PowerTransformer** → smooths the skew of selected continuous weather variables.
- **QuantileTransformer** → reshapes selected long-tailed environmental variables.
- **OneHotEncoder** → represents categorical time/context variables without fake numeric ordering.
- **SimpleImputer** → provides a consistent fallback for missing values.
- **No global scaler** → unnecessary for the tree-based XGBoost estimator.
- **Yeo–Johnson target transform** → stabilizes the target distribution while automatically converting predictions back to the original AQI scale.

The graphs above are intended to make those decisions visible rather than treating preprocessing as a black box.
